In [154]:
import numpy as np
import matplotlib.pyplot as plt

In [155]:
grid_h = 4
grid_w = 12

gridworld = np.zeros((4,12))

for h in range(grid_h):
    for w in range(grid_w):
        if(h == 3) and (w == 0 or w ==11):
            gridworld[h][w] = 0
        elif h != 3 and (w != 0 or w != 11):
            gridworld[h][w] = -1
        else:
            gridworld[h][w] = -100

terminal_state=[3,11]
initial_state = [3,0]

a_up, a_down, a_left, a_right = 0,1,2,3
actions = [a_up, a_down, a_left, a_right]

print(gridworld)

epsilon = 0.1
alpha = 0.5

gamma = 1 #for q_learning and expected_sarsa


action_symbols = [ '⬆', '⬇' , '⬅','⮕']

[[  -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.]
 [  -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.]
 [  -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.]
 [   0. -100. -100. -100. -100. -100. -100. -100. -100. -100. -100.    0.]]


In [156]:
def step(state, action):
    horizontol = None
    vertical = None
    if action == 0:
        horizontol = 0
        vertical = -1
    
    elif action == 1:
        horizontol = 0
        vertical = 1

    elif action == 2:
        horizontol = -1
        vertical = 0
    else:
        horizontol = 1
        vertical = 0

    next_state = [state[0] + vertical, state[1] + horizontol]

    if next_state[0] < 0:
        next_state[0] = 0
        reward = -100
        return (next_state, reward)
    elif next_state[0] > 3:
        next_state[0] = 3
        reward = -100
        return (next_state, reward)

    if next_state[1] < 0:
        next_state[1] = 0
        reward = -100
        return (next_state, reward)
    elif next_state[1] > 11:
        next_state[1] = 11
        reward = -100
        return (next_state, reward)

    reward = gridworld[next_state[0]][next_state[1]]

    return (next_state, reward)

def episode(init_start, terminal_state, epsilon, gamma, alpha, q_values):
    time = 0
    state = init_start
    end = terminal_state
    epsilon_var = epsilon
    action = None

    explore = np.random.rand()
    if(explore <= epsilon_var):
        action = np.random.choice(actions)
    else:
        max_indexes = np.where(q_values[state[0]][state[1]] == np.max(q_values[state[0]][state[1]]))[0]
        action_idx = np.random.choice(max_indexes)
        action = actions[action_idx]


    while state != end:

        next_state, reward = step(state, action)

        explore_next = np.random.rand()
        if(explore_next <= epsilon_var):
            action_next = np.random.choice(actions)
        else:
            max_indexes = max_indexes = np.where(q_values[next_state[0]][next_state[1]] == np.max(q_values[next_state[0]][next_state[1]]))[0]
            action_idx = np.random.choice(max_indexes)
            action_next = actions[action_idx]


        error = reward + \
            (gamma*q_values[next_state[0]][next_state[1]][action_next]) \
                  - q_values[state[0]][state[1]][action]
        update = q_values[state[0]][state[1]][action] + alpha * error

        q_values[state[0]][state[1]][action] = update

        state = next_state
        action = action_next

        time += 1

        epsilon_var = epsilon_var * time

    return q_values

In [157]:
def play(episodes, q_values):

   

    for _ in range(episodes):

        q_values = episode(initial_state, terminal_state, 0.1, 1, 0.5, q_values)

        

    return q_values

    
    

In [158]:
q_values_runs = np.zeros((4,12,4))

for run in range(50):
    q_values = np.zeros((4,12,4))

    q_values_runs += play(500, q_values)

q_values_runs = q_values_runs/50



In [159]:
def policy_from_value(step, grid_h, grid_w, action_space, optimal_values, action_symbols):
    """
    Extract a policy (with ties) from an optimal value function.

    Args:
        step            : transition function step(state, action)
        grid_dim        : grid size (int)
        action_space    : list of actions
        optimal_values  : (grid_dim, grid_dim) numpy array V*
        action_symbols  : dict mapping action index -> printable symbol

    Returns:
        policy_str : formatted string of greedy actions per state
                    (ties allowed, shown side-by-side)
    """

    policy_str = ""
    for row in range(grid_h):
        for col in range(grid_w):
            
            best = -9999999
            best_symbols = []
            for action in action_space:
                
                temp = q_values[row][col][action]
                if(temp > best):
                    best = temp
                    action_index = action_space.index(action)
                    best_symbols = [action_index]
                elif (temp == best):
                    action_index = action_space.index(action)
                    best_symbols.append(action_index)

            text = "".join(action_symbols[ba] for ba in best_symbols)
            
            policy_str += text + "\t"
        policy_str += "\n"
    return policy_str

policy_str = policy_from_value(step, grid_h,grid_w, actions, q_values_runs, action_symbols)
print(policy_str)

⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⬇	
⬆	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⬇	⬇	
⬆	⮕	⮕	⮕	⬆	⬆	⬆	⮕	⮕	⮕	⮕	⬇	
⬆	⬆	⮕	⮕	⮕	⬆	⬆	⮕	⮕	⮕	⮕	⬆⬇⬅⮕	



In [161]:
print(gridworld)

print(q_values[3][0])
print(q_values[3,10])

[[  -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.]
 [  -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.]
 [  -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.]
 [   0. -100. -100. -100. -100. -100. -100. -100. -100. -100. -100.    0.]]
[-8170.94576005 -8326.73937045 -8330.3595443  -8323.89109478]
[-3676.77471405 -2500.30607537 -4586.12948237     0.        ]
